# Exploración de la API Data360 del Banco Mundial

Este cuaderno tiene como objetivo explorar la API **Data360** del Banco Mundial, identificando su estructura, el volumen de información que maneja y realizando tres exploraciones visuales sobre indicadores clave de desarrollo.

**API Endpoint**: `https://data360api.worldbank.org`  
**Documentación**: Data360 Open API Spec

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración base
BASE_URL = "https://data360api.worldbank.org/data360"
sns.set_theme(style="whitegrid")

## 1. ¿Qué información trae esta API?

La API Data360 permite acceder a metadatos y series temporales de diversos conjuntos de datos del Banco Mundial, incluyendo el **World Development Indicators (WDI)**.

Comenzaremos explorando la cantidad de indicadores disponibles en la base de datos `WB_WDI`.

In [ ]:
def get_indicators(dataset_id="WB_WDI"):
    url = f"{BASE_URL}/indicators"
    params = {"datasetId": dataset_id}
    response = requests.get(url, params=params)
    return response.json()

indicators = get_indicators()
print(f"Total de indicadores en el dataset: {len(indicators)}")
print("Muestra de los primeros 10 indicadores:")
print(indicators[:10])

## 2. Volumen de Datos

Cada indicador puede contener datos para más de 200 países y regiones, con registros históricos que en algunos casos se remontan a 1960. Esto implica millones de puntos de datos disponibles.

Definiremos una función para recuperar datos específicos de un indicador y un país.

In [ ]:
def get_data(indicator_id, area_ids=None, database_id="WB_WDI"):
    url = f"{BASE_URL}/data"
    params = {
        "DATABASE_ID": database_id,
        "INDICATOR": indicator_id
    }
    if area_ids:
        params["REF_AREA"] = ",".join(area_ids)
    
    response = requests.get(url, params=params)
    data = response.json()
    return pd.DataFrame(data['value'])

# Probamos recuperando la población de Argentina
df_pop_arg = get_data("WB_WDI_SP_POP_TOTL", ["ARG"])
df_pop_arg['OBS_VALUE'] = pd.to_numeric(df_pop_arg['OBS_VALUE'])
df_pop_arg['TIME_PERIOD'] = pd.to_numeric(df_pop_arg['TIME_PERIOD'])
df_pop_arg.sort_values("TIME_PERIOD", inplace=True)
df_pop_arg.head()

## 3. Exploraciones de Datos

### Exploración 1: Crecimiento Poblacional en Argentina (1960-2023)
Visualizaremos cómo ha evolucionado la población total en Argentina a lo largo de las décadas.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_pop_arg['TIME_PERIOD'], df_pop_arg['OBS_VALUE'] / 1e6, marker='o', color= '#e05a1a')
plt.title('Evolución de la Población en Argentina', fontsize=16)
plt.xlabel('Año')
plt.ylabel('Población (Millones)')
plt.grid(True, alpha=0.3)
plt.show()

### Exploración 2: Comparativa de PBI per cápita (Argentina vs. Chile vs. Uruguay)
Analizaremos el PBI per cápita (PPP, current international $) para comparar economías vecinas.

In [ ]:
countries = ["ARG", "CHL", "URY"]
indicator_gdp = "WB_WDI_NY_GDP_PCAP_PP_CD"

df_gdp = get_data(indicator_gdp, countries)
df_gdp['OBS_VALUE'] = pd.to_numeric(df_gdp['OBS_VALUE'])
df_gdp['TIME_PERIOD'] = pd.to_numeric(df_gdp['TIME_PERIOD'])

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_gdp[df_gdp['TIME_PERIOD'] > 2000], x='TIME_PERIOD', y='OBS_VALUE', hue='REF_AREA')
plt.title('PBI per cápita (PPP) desde el año 2000', fontsize=16)
plt.ylabel('USD per cápita')
plt.xlabel('Año')
plt.legend(title='País')
plt.show()

### Exploración 3: Esperanza de Vida en el Cono Sur
Compararemos la esperanza de vida al nacer para ver las tendencias de salud y bienestar.

In [ ]:
indicator_life = "WB_WDI_SP_DYN_LE00_IN"
df_life = get_data(indicator_life, ["ARG", "CHL", "URY", "BRA"])
df_life['OBS_VALUE'] = pd.to_numeric(df_life['OBS_VALUE'])
df_life['TIME_PERIOD'] = pd.to_numeric(df_life['TIME_PERIOD'])

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_life[df_life['TIME_PERIOD'] > 1990], x='REF_AREA', y='OBS_VALUE', palette='Oranges')
plt.title('Distribución de Esperanza de Vida (1990-Presente)', fontsize=16)
plt.ylabel('Años de vida')
plt.xlabel('País')
plt.show()

## 4. Conclusiones

La API de Data360 es una herramienta sumamente potente para la visualización de datos de información, permitiendo extraer series de tiempo robustas para análisis comparativos globales. 

- **Riqueza de datos**: Más de 1.400 indicadores del WDI están integrados.
- **Granularidad**: Permite filtrado por país, región e indicadores específicos.
- **Potencial para visualización**: Facilita la creación de narrativas basadas en evidencia sobre desarrollo económico, salud y demografía.